# Model Risk Assessment for Credit Risk Model

## 1. Model Purpose

The model is developed to estimate borrower default risk and support credit risk ranking.

The model is intended to be used as a decision-support tool for manual credit review and risk segmentation. It should not be used as a fully automated loan rejection system without additional governance, human oversight, and compliance review.

Intended use cases:
- Rank borrowers by predicted default risk
- Support manual review prioritization
- Identify high-risk borrower segments
- Provide input for threshold-based risk strategy analysis

Out-of-scope use cases:
- Fully automated loan rejection
- Pricing decisions without business review
- Application to customer groups not represented in the development sample

## 2. Model Scope and Assumptions

The model is applicable to borrower profiles that are similar to the training data population.

Key assumptions:
- The relationship between borrower characteristics and default risk remains relatively stable.
- The input data follows similar definitions and quality standards as the development dataset.
- Missing values and abnormal values are handled consistently before scoring.
- The model is used together with business rules and human review.

Limitations:
- The model may become less reliable under major macroeconomic changes.
- The model may not generalize to new loan products or new customer acquisition channels.
- The dataset does not include all possible behavioral, macroeconomic, or alternative credit variables.

## 3. Data Risk Assessment

Main data risks identified in the project:

| Risk Area | Observation | Potential Impact | Mitigation |
|---|---|---|---|
| Missing values | MonthlyIncome and NumberOfDependents contain missing values | May reduce model reliability for affected groups | Median imputation with missing indicators |
| Abnormal values | Age = 0 and delinquency values such as 96/98 are treated as abnormal | May distort model coefficients or risk ranking | Replace abnormal values with missing values before imputation |
| Class imbalance | Default cases are much fewer than non-default cases | Accuracy may be misleading; recall and PR-AUC are needed | Use ROC-AUC, PR-AUC, recall, and decile analysis |
| Sample representativeness | Dataset may not fully represent future borrower populations | Model may degrade when customer mix changes | Monitor PSI and segment-level performance |

## 4. Performance Risk Assessment

The final calibrated Histogram Gradient Boosting model achieved the following performance on the test set:

| Metric | Test Result | Interpretation |
|---|---:|---|
| ROC-AUC | 0.8685 | Strong discriminatory power |
| Average Precision | 0.4055 | Useful ranking performance under class imbalance |
| Brier Score | 0.0487 | Reasonable probability calibration |
| Portfolio Default Rate | 6.68% | Baseline default level |

The model shows stronger discriminatory performance than the baseline logistic regression model. However, model performance should be monitored regularly after deployment, especially if borrower population or economic conditions change.

## 5. Threshold and Business Risk Assessment

A 10% review-rate threshold was selected based on validation set performance.

At the selected threshold of 0.1765, the model performance on the test set was:

| Metric | Test Result |
|---|---:|
| Actual review rate | 10.13% |
| Precision among reviewed borrowers | 36.77% |
| Recall / captured defaulters | 55.71% |
| False positive rate | 6.86% |
| True positives | 1,117 |
| False positives | 1,921 |
| False negatives | 888 |

Business interpretation:

The threshold captures more than half of observed defaulters while reviewing about 10% of borrowers. This makes the model suitable for prioritizing manual review. However, the false positive count indicates that many non-default borrowers would still be flagged as high risk. Therefore, the model should support human review rather than automatic rejection.

## 6. Stability Risk Assessment

Population Stability Index (PSI) was used to compare the distribution of predicted risk scores between training and test samples.

| Metric | Value | Interpretation |
|---|---:|---|
| PSI | 0.00059 | No material population shift detected |

The PSI result suggests that the score distribution is stable between the training and test samples. However, this is an internal validation result and does not guarantee future stability after deployment. The model should be monitored using monthly or quarterly PSI after launch.

## 7. Interpretability Risk Assessment

The model uses a tree-based machine learning method, which provides better predictive performance than logistic regression but is less directly interpretable.

Interpretability risks:
- Business users may find it difficult to understand why a borrower is classified as high risk.
- Individual-level decisions require clear explanation before being used in customer-impacting workflows.
- Feature importance alone may not fully explain individual predictions.

Recommended mitigation:
- Add SHAP-based global and local explanations.
- Provide top risk drivers for each high-risk borrower.
- Use logistic regression as a benchmark model for interpretability.
- Avoid using the model as the sole basis for rejection decisions.

## 8. Fairness and Segment Risk Assessment

The model should be evaluated across key borrower segments to identify whether performance differs materially across groups.

Suggested segment checks:
- Age groups
- Income availability / income bands
- Number of dependents
- Delinquency history groups

Key questions:
- Does the model produce much higher false positive rates for certain groups?
- Does recall differ significantly across borrower segments?
- Are any variables acting as proxies for sensitive or protected characteristics?

If large performance gaps are observed across segments, the model should be reviewed before deployment.

In [1]:
def segment_default_rate(data, segment_col, target_col="actual_result"):
    return (
        data.groupby(segment_col)
        .agg(
            customer_count=(target_col, "count"),
            default_rate=(target_col, "mean")
        )
        .reset_index()
    )

## 9. Post-Deployment Monitoring Plan

Recommended monitoring indicators:

| Monitoring Area | Metric | Review Frequency | Trigger |
|---|---|---|---|
| Discrimination | ROC-AUC / KS | Monthly or quarterly | AUC drops by more than 5% |
| Stability | PSI | Monthly | PSI > 0.20 |
| Calibration | Brier Score / calibration curve | Quarterly | Significant deviation from observed default rate |
| Business impact | Review rate / approval rate / bad rate | Monthly | Material deviation from expected range |
| Segment fairness | Segment-level false positive rate and recall | Quarterly | Large unexplained group differences |
| Data quality | Missing rate / abnormal value rate | Monthly | Sharp increase in missing or abnormal values |

## 10. Overall Validation Opinion

Based on the current validation results, the model demonstrates strong discriminatory performance and stable score distribution on the test sample.

Validation conclusion:

The model may be approved for limited use as a credit risk ranking and manual review support tool, subject to the following conditions:

- The model should not be used as the sole basis for automatic loan rejection.
- Monthly performance and PSI monitoring should be implemented.
- Segment-level performance should be reviewed before production deployment.
- SHAP-based interpretability analysis should be added before customer-level decisioning.
- The model should be reviewed if PSI exceeds 0.20, AUC declines by more than 5%, or observed bad rate materially deviates from expectations.

Overall recommendation: Approve with conditions.